In [ ]:
# 합치기 코드
import json

FILE_150 = "RPT_train_150.json"  # ← 기존 150건
FILE_AUG = "RPT_AUG_ONLY.json"          # 증강 50건
OUT      = "RPT_train_200.json"         # 합친 결과

# 로드 (list 또는 {"data":[...]} 둘 다 처리)
def load_list(path):
    d = json.load(open(path, encoding="utf-8"))
    return d["data"] if isinstance(d, dict) and "data" in d else d

base = load_list(FILE_150)
aug  = load_list(FILE_AUG)

merged = base + aug

# id 중복 체크 (중요!)
ids = [r.get("id") for r in merged if r.get("id")]
dup = set([x for x in ids if ids.count(x) > 1])
if dup:
    print(f"⚠️ id 중복 {len(dup)}개:", list(dup)[:10])
else:
    print("id 중복 없음 ✅")

json.dump(merged, open(OUT, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"{len(base)} + {len(aug)} = {len(merged)}건 → {OUT}")

⚠️ id 중복 11개: ['RPT-03-02', 'RPT-01-03', 'RPT-02-03-01', 'RPT-02-04-01', 'RPT-02-01', 'RPT-02-06', 'RPT-02-05', 'RPT-01-02', 'RPT-01-01', 'RPT-02-02']
150 + 50 = 200건 → RPT_train_200.json


In [1]:
# 원본 개수 확인
import json
from collections import Counter

PATHS = {
    "CON": {"train": "CON_train_200.json", "valid": "CON_valid_15.json"},
    "PEP": {"train": "PEP_train_200.json", "valid": "PEP_valid_15.json"},
    "RPT": {"train": "RPT_train_200.json", "valid": "RPT_valid_15.json"},
}

def _load(path):
    d = json.load(open(path, encoding='utf-8'))
    return d['data'] if isinstance(d, dict) and 'data' in d else d

print("="*55)
print(f"{'태스크':<6}{'파일':<8}{'개수':<8}{'label 분포'}")
print("="*55)
EXPECT = {"train": 150, "valid": 15}
ok = True
for task, p in PATHS.items():
    for kind in ["train", "valid"]:
        data = _load(p[kind])
        n = len(data)
        # label 추출 (CON: label / PEP·RPT: output.label)
        labs = Counter(
            r.get("label", r.get("output", {}).get("label", "?")) for r in data
        )
        flag = "" if n == EXPECT[kind] else f"  ⚠️ (기대 {EXPECT[kind]})"
        if n != EXPECT[kind]: ok = False
        print(f"{task:<6}{kind:<8}{n:<8}{dict(labs)}{flag}")
print("="*55)
print("✅ 개수 정상" if ok else "❌ 개수 이상 — 파일 확인 필요 (변환 중단 권장)")

태스크   파일      개수      label 분포
CON   train   200     {'검토': 57, '일치': 56, '불일치': 87}  ⚠️ (기대 150)
CON   valid   15      {'검토': 3, '일치': 5, '불일치': 7}
PEP   train   201     {'불가': 89, '검토': 51, '충족': 61}  ⚠️ (기대 150)
PEP   valid   14      {'불가': 8, '검토': 2, '충족': 4}  ⚠️ (기대 15)
RPT   train   200     {'불가': 90, '충족': 70, '검토': 40}  ⚠️ (기대 150)
RPT   valid   15      {'불가': 6, '충족': 4, '검토': 5}
❌ 개수 이상 — 파일 확인 필요 (변환 중단 권장)


In [2]:
import json
import random

# ── train + valid 경로 ──
PATHS = {
    "CON": {"train": "CON_train_200.json", "valid": "CON_valid_15.json"},
    "PEP": {"train": "PEP_train_200.json", "valid": "PEP_valid_15.json"},
    "RPT": {"train": "RPT_train_200.json", "valid": "RPT_valid_15.json"},
}

def load(path):
    d = json.load(open(path, encoding='utf-8'))
    return d['data'] if isinstance(d, dict) and 'data' in d else d

def load_prompt(path):
    return open(path, encoding='utf-8').read().strip()

CON_SYSTEM = load_prompt('CON_system_prompt.txt')
PEP_SYSTEM = load_prompt('PEP_system_prompt.txt')
RPT_SYSTEM = load_prompt('RPT_system_prompt.txt')

# ── CON: {category, clause_text, refs} → {판정, 사고과정, 근거} ──
def con_to_messages(item):
    user = {"category": item["category"], "clause_text": item["clause_text"], "refs": item["refs"]}
    asst = {"판정": item["label"], "사고과정": item.get("사고과정", ""), "근거": item["근거"]}
    return {
        "messages": [
            {"role": "system", "content": CON_SYSTEM},
            {"role": "user", "content": json.dumps(user, ensure_ascii=False)},
            {"role": "assistant", "content": json.dumps(asst, ensure_ascii=False)},
        ],
        "meta": {"id": item["id"], "task": "CON", "label": item["label"]},
    }

# ── PEP: {criteria, rfp_excerpt, pep_excerpt} → {label, eval} ──
def pep_to_messages(item):
    inp = item["input"]
    user = {"criteria": inp["criteria"], "rfp_excerpt": inp["rfp_excerpt"], "pep_excerpt": inp["pep_excerpt"]}
    asst = {"label": item["output"]["label"], "eval": item["output"]["eval"]}
    return {
        "messages": [
            {"role": "system", "content": PEP_SYSTEM},
            {"role": "user", "content": json.dumps(user, ensure_ascii=False)},
            {"role": "assistant", "content": json.dumps(asst, ensure_ascii=False)},
        ],
        "meta": {"id": item["id"], "task": "PEP", "label": item["output"]["label"]},
    }

# ── RPT: {criteria, pep_excerpt, rpt_excerpt} → {label, eval} ──
def rpt_to_messages(item):
    inp = item["input"]
    user = {"criteria": inp["criteria"], "pep_excerpt": inp["pep_excerpt"], "rpt_excerpt": inp["rpt_excerpt"]}
    asst = {"label": item["output"]["label"], "eval": item["output"]["eval"]}
    return {
        "messages": [
            {"role": "system", "content": RPT_SYSTEM},
            {"role": "user", "content": json.dumps(user, ensure_ascii=False)},
            {"role": "assistant", "content": json.dumps(asst, ensure_ascii=False)},
        ],
        "meta": {"id": item.get("seed_id", item.get("id")), "task": "RPT", "label": item["output"]["label"]},
    }

CONV = {"CON": con_to_messages, "PEP": pep_to_messages, "RPT": rpt_to_messages}

def write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

# ── 변환 (train + valid 각각) ──
all_train, all_valid = [], []
for task, p in PATHS.items():
    conv = CONV[task]
    tr = [conv(i) for i in load(p["train"])]
    va = [conv(i) for i in load(p["valid"])]
    write_jsonl(f'kanana_{task.lower()}_train.jsonl', tr)
    write_jsonl(f'kanana_{task.lower()}_valid.jsonl', va)
    all_train += tr
    all_valid += va
    print(f"{task}: train {len(tr)} / valid {len(va)}")

# ── 합본 (train만 셔플, valid는 유지) ──
random.seed(42)
random.shuffle(all_train)
write_jsonl('kanana_all_train.jsonl', all_train)
write_jsonl('kanana_all_valid.jsonl', all_valid)
print(f"\nALL: train {len(all_train)} / valid {len(all_valid)}")

CON: train 200 / valid 15
PEP: train 201 / valid 14
RPT: train 200 / valid 15

ALL: train 601 / valid 44


In [ ]:
import json
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("kakaocorp/kanana-1.5-8b-instruct-2505")

rows = [json.loads(l) for l in open('kanana_all_train.jsonl', encoding='utf-8')]
lens = []
for r in rows:
    text = tok.apply_chat_template(r['messages'], tokenize=False)
    lens.append(len(tok(text)['input_ids']))
lens.sort()
print(f"토큰 최소/중앙/평균/최대: {lens[0]}/{lens[len(lens)//2]}/{sum(lens)//len(lens)}/{lens[-1]}")
for ml in [2048, 4096, 6144, 8192]:
    over = sum(1 for n in lens if n > ml)
    print(f"  max_length {ml}: {over}건 잘림 ({over/len(lens)*100:.1f}%)")

config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/66.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

토큰 최소/중앙/평균/최대: 1857/2635/2739/4230
  max_length 2048: 588건 잘림 (97.8%)
  max_length 4096: 5건 잘림 (0.8%)
  max_length 6144: 0건 잘림 (0.0%)
  max_length 8192: 0건 잘림 (0.0%)


In [ ]:
import json
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("kakaocorp/kanana-1.5-8b-instruct-2505")

# CON만 뽑기
rows = [json.loads(l) for l in open('kanana_all_train.jsonl', encoding='utf-8')]
con_rows = [r for r in rows if r['meta']['task'] == 'CON']
print(f"CON: {len(con_rows)}건")

# 현재(refs 2개 기준) CON 토큰
lens = []
for r in con_rows:
    text = tok.apply_chat_template(r['messages'], tokenize=False)
    lens.append(len(tok(text)['input_ids']))
lens.sort()
print(f"[CON 현재] 최소/중앙/평균/최대: {lens[0]}/{lens[len(lens)//2]}/{sum(lens)//len(lens)}/{lens[-1]}")
for ml in [2048, 4096, 6144, 8192]:
    over = sum(1 for n in lens if n > ml)
    print(f"  max_length {ml}: {over}건 잘림 ({over/len(lens)*100:.1f}%)")

# ── refs 1개당 평균 토큰 (chunk_text 크기) 추정 ──
ref_tokens = []
for r in con_rows:
    user = json.loads(r['messages'][1]['content'])   # user payload
    for rf in user.get('refs', []):
        ref_tokens.append(len(tok(rf.get('chunk_text',''))['input_ids']))
if ref_tokens:
    avg_ref = sum(ref_tokens)//len(ref_tokens)
    print(f"\n[refs 1개당 평균 토큰] {avg_ref} (조문 본문)")
    print(f"→ refs 2개→3개로 늘리면 건당 약 +{avg_ref} 토큰")
    print(f"→ CON 최대 예상: {lens[-1]} + {avg_ref} ≈ {lens[-1]+avg_ref}")

CON: 35건
[CON 현재] 최소/중앙/평균/최대: 1300/2019/1993/2748
  max_length 2048: 16건 잘림 (45.7%)
  max_length 4096: 0건 잘림 (0.0%)
  max_length 6144: 0건 잘림 (0.0%)
  max_length 8192: 0건 잘림 (0.0%)

[refs 1개당 평균 토큰] 579 (조문 본문)
→ refs 2개→3개로 늘리면 건당 약 +579 토큰
→ CON 최대 예상: 2748 + 579 ≈ 3327
